# AUModel — Phase 1: Train Turkish BPE Tokenizer

This notebook trains a 64k-vocabulary SentencePiece BPE tokenizer on Turkish Wikipedia.

**Estimated total runtime**: 35–55 minutes on Colab CPU (T4 GPU not needed for this phase).

| Step | Action | Est. Time |
|------|--------|-----------|
| 1 | Install deps + mount Drive | < 2 min |
| 2 | Download corpus (`--download`) | 10–15 min |
| 3 | Train tokenizer (`--train`) | 25–35 min |
| 4 | Validate + copy to Drive | < 2 min |

---
## Section 1: Install Dependencies & Mount Google Drive

In [ ]:
# Install required packages
# Expected: all packages install without errors
!pip install sentencepiece>=0.1.99 datasets>=2.18.0 tqdm>=4.66.0 -q
print("[OK] Dependencies installed")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/aumodel_checkpoints/tokenizer'

import os
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"[OK] Drive mounted. Artifacts will be saved to: {DRIVE_DIR}")

In [ ]:
# Clone the repo on the Colab runtime filesystem (/content/)
# Works the same whether you open this notebook in browser Colab OR via VS Code Colab extension
# (Colab runtime has its own filesystem — separate from your local machine)
import os

REPO_URL = 'https://github.com/aykanugur/AU_MODEL.git'
REPO_BRANCH = '001-turkish-tokenizer'
REPO_DIR = '/content/AU_MODEL'

if not os.path.exists(REPO_DIR):
    !git clone -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}
    print("[OK] Repo cloned")
else:
    print("[OK] Repo already present — pulling latest")
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
!git log --oneline -3
print(f"[OK] Working dir: {os.getcwd()}")

---
## Section 1.5: Pre-Training Dry-Run Verification (Run Before Downloading Full Corpus)

Fetches **50 docs** from the real dataset, trains a **500-vocab mini tokenizer** with identical parameters, and asserts all IDs and Turkish char coverage are correct.

**If all assertions pass → safe to run Section 2 & 3 for real.**  
Expected runtime: ~60 seconds.

In [ ]:
"""
DRY-RUN VERIFICATION
====================
Tests the exact same SPM parameters that will be used in real training,
but on 50 real Wikipedia docs + vocab_size=500.
All assertions must pass before proceeding to Section 2.
"""
import os, sys, tempfile, unicodedata, re
import sentencepiece as spm

PASSES = []
FAILS = []

def check(name, condition, detail=""):
    if condition:
        PASSES.append(name)
        print(f"  [PASS] {name}")
    else:
        FAILS.append(name)
        print(f"  [FAIL] {name}  ← {detail}")

print("=" * 60)
print("STEP 1: Fetch 50 docs from wikimedia/wikipedia (20231101.tr)")
print("=" * 60)

from datasets import load_dataset
from tqdm import tqdm

dataset = load_dataset("wikimedia/wikipedia", "20231101.tr", split="train", streaming=True)

docs = []
for doc in tqdm(dataset, total=50, desc="Fetching"):
    text = doc.get("text", "") or ""
    text = unicodedata.normalize("NFC", text)
    if len(text) >= 200:
        docs.append(text.replace("\n", " "))
    if len(docs) >= 50:
        break

check("dataset_accessible",       len(docs) > 0,        f"got {len(docs)} docs")
check("text_field_exists",        all("text" in d for d in [doc] for doc in [dataset.__iter__().__next__()]), "")
check("nfc_applied",              len(docs) == 50,      f"only {len(docs)} docs after NFC filter")

# Write mini corpus
corpus_file = "/tmp/dry_run_corpus.txt"
with open(corpus_file, "w", encoding="utf-8") as f:
    for d in docs:
        f.write(d + "\n")

check("corpus_written",           os.path.exists(corpus_file),    "file not found")
check("corpus_not_empty",         os.path.getsize(corpus_file) > 1024, f"{os.path.getsize(corpus_file)} bytes")

print(f"\n  Corpus: {len(docs)} docs, {os.path.getsize(corpus_file)/1024:.1f} KB\n")

print("=" * 60)
print("STEP 2: Train mini SPM (vocab=500, same params as real run)")
print("=" * 60)

model_prefix = "/tmp/dry_run_tokenizer"

try:
    spm.SentencePieceTrainer.train(
        input=corpus_file,
        model_prefix=model_prefix,
        model_type="bpe",
        vocab_size=500,
        character_coverage=0.9999,
        normalization_rule_name="identity",
        byte_fallback=True,
        input_sentence_size=10_000,
        shuffle_input_sentence=True,
        pad_id=0,
        unk_id=1,
        bos_id=2,
        eos_id=3,
        user_defined_symbols=["[SYSTEM]", "[USER]", "[ASSISTANT]", "[SEP]"],
    )
    check("spm_train_completed", True)
except Exception as e:
    check("spm_train_completed", False, str(e))
    print("\n[FATAL] Training failed — check params above."); raise

sp = spm.SentencePieceProcessor()
sp.load(model_prefix + ".model")

print(f"\n  Model loaded. Vocab size: {sp.get_piece_size()}\n")

print("=" * 60)
print("STEP 3: Verify special token IDs")
print("=" * 60)

check("pad_id==0",         sp.pad_id() == 0,                  f"got {sp.pad_id()}")
check("unk_id==1",         sp.unk_id() == 1,                  f"got {sp.unk_id()}")
check("bos_id==2",         sp.bos_id() == 2,                  f"got {sp.bos_id()}")
check("eos_id==3",         sp.eos_id() == 3,                  f"got {sp.eos_id()}")
check("[SYSTEM]==4",       sp.piece_to_id("[SYSTEM]") == 4,   f"got {sp.piece_to_id('[SYSTEM]')}")
check("[USER]==5",         sp.piece_to_id("[USER]") == 5,     f"got {sp.piece_to_id('[USER]')}")
check("[ASSISTANT]==6",    sp.piece_to_id("[ASSISTANT]") == 6,f"got {sp.piece_to_id('[ASSISTANT]')}")
check("[SEP]==7",          sp.piece_to_id("[SEP]") == 7,      f"got {sp.piece_to_id('[SEP]')}")

# All 8 IDs must be distinct
ids = [sp.pad_id(), sp.unk_id(), sp.bos_id(), sp.eos_id(),
       sp.piece_to_id("[SYSTEM]"), sp.piece_to_id("[USER]"),
       sp.piece_to_id("[ASSISTANT]"), sp.piece_to_id("[SEP]")]
check("all_special_ids_distinct", len(set(ids)) == 8, f"duplicates in {ids}")

print()
print("=" * 60)
print("STEP 4: Verify Turkish chars (12 exact Unicode chars)")
print("=" * 60)

TURKISH_CHARS = ["\u00e7","\u011f","\u0131","\u0130","\u00f6","\u015f",
                 "\u00fc","\u00dc","\u00d6","\u00c7","\u011e","\u015e"]
BYTE_RE = re.compile(r"^<0x[0-9A-Fa-f]+>$")

native_count = 0
for ch in TURKISH_CHARS:
    pid = sp.piece_to_id(ch)
    piece = sp.id_to_piece(pid)
    is_native = (pid != sp.unk_id()) and not BYTE_RE.match(piece) and piece == ch
    if is_native:
        native_count += 1
        print(f"  [PASS] '{ch}' (U+{ord(ch):04X}) → ID {pid}")
    else:
        print(f"  [WARN] '{ch}' (U+{ord(ch):04X}) → piece='{piece}' ID={pid}  (byte-fallback or UNK — OK in 500-vocab, will be native in 64k)")

# In a 500-vocab dry run some rare chars may still be byte-fallback — that's expected.
# Real threshold check is in validate_tokenizer.py against the 64k model.
print(f"\n  {native_count}/12 chars native at vocab=500 (all 12 expected at vocab=64k)")

print()
print("=" * 60)
print("STEP 5: Encode/decode round-trip")
print("=" * 60)

test_strings = ["merhaba", "çalışmak", "Türkiye güzel bir ülkedir.", "atma", "öğrenci"]
for s in test_strings:
    decoded = sp.decode(sp.encode(s, out_type=int))
    ok = decoded == s
    check(f"roundtrip:{s[:15]}", ok, f"got '{decoded}'")

print()
print("=" * 60)
print("FINAL RESULT")
print("=" * 60)

if FAILS:
    print(f"\n  FAIL — {len(FAILS)} check(s) failed:")
    for f in FAILS:
        print(f"    - {f}")
    print("\n  Do NOT proceed to Section 2 until failures are resolved.")
else:
    print(f"\n  ALL {len(PASSES)} CHECKS PASSED")
    print("  Safe to proceed to Section 2 (--download) and Section 3 (--train)")


---
## Section 2: Download Turkish Wikipedia Corpus

Downloads `wikimedia/wikipedia 20231101.tr` via HuggingFace, applies NFC normalization,
filters documents < 200 chars, and writes one document per line to
`data/raw/tokenizer_corpus.txt`.

**Expected output**:
```
[INFO] Downloading wikimedia/wikipedia (20231101.tr) ...
[INFO] Wrote 1,180,000 documents (45,000 skipped) -> data/raw/tokenizer_corpus.txt
[INFO] Corpus size: 7400.0 MB [OK]
```
**Expected duration**: ~10–15 minutes (network dependent)

In [ ]:
import os
corpus_path = 'data/raw/tokenizer_corpus.txt'
if os.path.exists(corpus_path):
    size_mb = os.path.getsize(corpus_path) / 1024 / 1024
    print(f"[INFO] Corpus already exists ({size_mb:.0f} MB). Skipping download.")
    SKIP_DOWNLOAD = True
else:
    print("[INFO] Corpus not found — will download.")
    SKIP_DOWNLOAD = False

In [ ]:
if not SKIP_DOWNLOAD:
    !python tokenizer/train_tokenizer.py --download
else:
    print("[SKIP] Download skipped (corpus already present)")

In [ ]:
# Verify corpus
import os
corpus_path = 'data/raw/tokenizer_corpus.txt'
assert os.path.exists(corpus_path), f"Corpus not found: {corpus_path}"
size_mb = os.path.getsize(corpus_path) / 1024 / 1024
print(f"[OK] Corpus: {corpus_path} ({size_mb:.0f} MB)")

# Show first 3 lines
with open(corpus_path, encoding='utf-8') as f:
    for i, line in enumerate(f):
        print(f"  Line {i+1}: {line[:80]}...")
        if i >= 2:
            break

---
## Section 3: Train the Tokenizer

Trains a 64k-vocabulary BPE SentencePiece model with locked parameters:
- `vocab_size=64000`, `model_type=bpe`, `character_coverage=0.9999`
- `normalization_rule_name=identity`, `byte_fallback=True`
- `input_sentence_size=10_000_000`, `random_seed=42`
- Special tokens: `<pad>=0, <unk>=1, <s>=2, </s>=3, [SYSTEM]=4, [USER]=5, [ASSISTANT]=6, [SEP]=7`

**Expected output**:
```
[INFO] Training SentencePiece BPE tokenizer (vocab_size=64,000) ...
[INFO] Model saved: tokenizer/turkish_bpe.model [OK]
[INFO] Vocab saved: tokenizer/turkish_bpe.vocab [OK]
[INFO] Vocab size verified: 64,000 [OK]
```
**Expected duration**: ~25–35 minutes on Colab CPU

In [ ]:
import os
model_path = 'tokenizer/turkish_bpe.model'
if os.path.exists(model_path):
    size_mb = os.path.getsize(model_path) / 1024 / 1024
    print(f"[INFO] Model already exists ({size_mb:.1f} MB). Use --force to retrain.")
    SKIP_TRAIN = True
else:
    print("[INFO] Model not found — will train.")
    SKIP_TRAIN = False

In [ ]:
if not SKIP_TRAIN:
    !python tokenizer/train_tokenizer.py --train
else:
    print("[SKIP] Training skipped (model already present)")
    print("       Pass --force to retrain: !python tokenizer/train_tokenizer.py --train --force")

In [ ]:
# Verify model file
import sentencepiece as spm
import os

model_path = 'tokenizer/turkish_bpe.model'
assert os.path.exists(model_path), f"Model not found: {model_path}"

sp = spm.SentencePieceProcessor()
sp.load(model_path)

assert sp.get_piece_size() == 64000, f"Expected 64000 vocab, got {sp.get_piece_size()}"

print(f"[OK] Model loaded: {model_path}")
print(f"[OK] Vocab size: {sp.get_piece_size():,}")
print(f"[OK] pad_id={sp.pad_id()}, unk_id={sp.unk_id()}, bos_id={sp.bos_id()}, eos_id={sp.eos_id()}")
print(f"[OK] Quick encode test: {sp.encode('merhaba dunya')}")

---
## Section 4: Validate Tokenizer & Copy Artifacts to Drive

Runs 4 validation checks:
- **fertility**: avg tokens/word <= 1.4 (sampled 10,000 sentences)
- **roundtrip**: encode->decode lossless for 100 test strings
- **turkish_chars**: all 12 Turkish chars have dedicated vocabulary entries
- **special_tokens**: all 8 special token IDs are correct and distinct

**Expected exit code**: 0 (all pass)

**Expected output**:
```
[INFO] Model loaded: tokenizer/turkish_bpe.model (vocab_size=64,000)

Check              Status       Value  Threshold
------------------------------------------------
fertility          [PASS]      1.2300     1.4000
roundtrip          [PASS]    100.0000   100.0000
turkish_chars      [PASS]     12.0000    12.0000
special_tokens     [PASS]      8.0000     8.0000

All checks passed.
```

In [ ]:
!python tokenizer/validate_tokenizer.py
print(f"\nValidation exit code: {__import__('subprocess').run(['python','tokenizer/validate_tokenizer.py'], capture_output=True).returncode}")

In [ ]:
# Smoke test the Tokenizer wrapper (US3 interface)
from tokenizer import Tokenizer

tok = Tokenizer('tokenizer/turkish_bpe.model')

assert tok.vocab_size == 64000
assert tok.pad_id == 0
assert tok.unk_id == 1
assert tok.bos_id == 2
assert tok.eos_id == 3
assert tok.system_id == 4
assert tok.user_id == 5
assert tok.assistant_id == 6
assert tok.sep_id == 7

# Round-trip
test = 'merhaba dunya'
assert tok.decode(tok.encode(test)) == test, "Round-trip failed!"

# BOS/EOS
ids_with_bos_eos = tok.encode(test, add_bos=True, add_eos=True)
assert ids_with_bos_eos[0] == tok.bos_id
assert ids_with_bos_eos[-1] == tok.eos_id

# Empty string
assert tok.encode('') == []

print(f"[OK] Tokenizer: {tok}")
print(f"[OK] encode('merhaba dunya') -> {tok.encode('merhaba dunya')}")
print(f"[OK] All Tokenizer assertions passed")

In [ ]:
# Copy model artifacts to Google Drive for persistence across Colab sessions
import shutil
import os

artifacts = [
    ('tokenizer/turkish_bpe.model', f'{DRIVE_DIR}/turkish_bpe.model'),
    ('tokenizer/turkish_bpe.vocab', f'{DRIVE_DIR}/turkish_bpe.vocab'),
]

for src, dst in artifacts:
    if os.path.exists(src):
        shutil.copy2(src, dst)
        size_mb = os.path.getsize(dst) / 1024 / 1024
        print(f"[OK] Copied {src} -> {dst} ({size_mb:.1f} MB)")
    else:
        print(f"[WARN] Artifact not found: {src} — skipping")

print(f"\nArtifacts saved to: {DRIVE_DIR}")
print("Phase 1 (Tokenizer) complete!")